In [ ]:
import configparser
import os
import sys
from  datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Setup print length for pandas dataframes
pd.set_option('display.max_colwidth', None)
pd.set_option('expand_frame_repr', False)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
sys.path.append(str(Path.cwd().parent))

from etrade_client.auth import EtradeAuthorization, ensure_credentials
from etrade_client.accounts import Accounts
from etrade_client.market import Market
from etrade_client.option_chain import OptionChain, get_options_research, extend_options_research
from etrade_client.research import OptionsResearchDelta

In [ ]:
# Read config file
config = configparser.ConfigParser()
ini_file = Path.cwd() / "config.ini"
config.read(ini_file);

In [ ]:
session, base_url = EtradeAuthorization(config["DEFAULT"], open_browser=False)
market = Market(session, base_url)

In [ ]:
# Research ticker
ticker = "GOOG"
qty = 100
cost_basis = market.quote(ticker)['lastPrice'][0]
opt_type = "CALL"

chain = OptionChain(session, base_url, ticker, as_dataframe=True)
chain.getExpiryDates()[:4]

In [ ]:
dates = ["11-20-2026"]
# dates = ["05-08-2026", "05-15-2026", "05-29-2026"]
exp_chain = {(datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days: chain.view(d) for d in dates}

In [ ]:
print("Ticker:", ticker)
print("Cost Basis:", cost_basis)
for d, chain_df in exp_chain.items():
    res1 = get_options_research(chain_df, opt_type)
    chain_research = extend_options_research(res1, cost_basis, d, qty)
    print(f"Days to Expiry: {d}")
    print(chain_research)

In [ ]:
# Research sector: OIL
tickers = ["OXY", "CVX", "XOM", "PSX"]
cost_bases = [market.quote(t)['lastPrice'][0] for t in tickers]
opt_type = "CALL"
qty = 100

chains = {t: OptionChain(session, base_url, t, as_dataframe=True) for t in tickers}

chain_research_df = []
for ticker, chain in chains.items():
    cost_basis = cost_bases[tickers.index(ticker)]
    print(f"\n    !!!!    Ticker: {ticker}  - Cost Basis: {cost_basis}    !!!!\n")
    # dates = ["06-18-2026", "07-17-2026"]
    dates = chain.getExpiryDates()["expiryDate"]
    # dates = dates[2:]
    # print("All Expiry Dates:", dates)
    # No longer than a 1.5 month to expiry
    dates45 = [d for d in dates if 0 < (datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days <= 45]
    exp_chain = {(datetime.strptime(d, "%m-%d-%Y") - datetime.today()).days: chain.view(d) for d in dates45}
    ret_chain_df = []
    for d, chain_df in exp_chain.items():
        res1 = get_options_research(chain_df, opt_type=opt_type, delta_min=0.05, delta_max=0.3)
        print(f"Processing chain for {ticker} with {d} days to expiry")
        chain_research = extend_options_research(res1, cost_basis, d, qty)
        date = dates45[list(exp_chain.keys()).index(d)]
        print(f"Days to Expiry: {d}, date: {date}")
        print(chain_research)
        ret_chain_df.append(chain_research)
    chain_research_df.append(ret_chain_df)